[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neeraj-x0/idm_vton/blob/main/IDM_VTON_Colab.ipynb)

# IDM-VTON — Virtual Try-On (Google Colab)

Run **[IDM-VTON](https://github.com/yisol/IDM-VTON)** locally in Colab: upload a person photo + garment image and generate a realistic try-on result.

| Resource | Link |
|---|---|
| Paper | [arXiv:2403.05139](https://arxiv.org/abs/2403.05139) |
| Official demo (no setup) | [Hugging Face Space](https://huggingface.co/spaces/yisol/IDM-VTON) |
| Model weights | [yisol/IDM-VTON](https://huggingface.co/yisol/IDM-VTON) |

## Quick start
1. **Runtime → Change runtime type → GPU** (A100 or L4 strongly recommended; T4 often OOMs).
2. **Runtime → Run all** (first run downloads ~8 GB and takes ~10–15 min).
3. Open the **Gradio public URL** from the last cell and upload your images.

> **VRAM note:** IDM-VTON is SDXL-based and needs **~20 GB VRAM** at 768×1024. Colab T4 (16 GB) frequently crashes. Use Colab Pro/Pro+ for A100/L4, or try the hosted [HF demo](https://huggingface.co/spaces/yisol/IDM-VTON) instead.

In [ ]:
# @title 1) GPU check & memory settings
import os
import subprocess
import sys

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/content/IDM-VTON" if IN_COLAB else "./IDM-VTON"

if not os.path.exists("/content"):
    os.makedirs(WORK_DIR, exist_ok=True)

try:
    import torch
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
        print(f"GPU: {name} ({vram_gb:.1f} GB VRAM)")
        if vram_gb < 20:
            print(
                "\n⚠️  Warning: <20 GB VRAM detected. Inference may OOM on T4/L4.\n"
                "   • Prefer A100 (40 GB) via Colab Pro+\n"
                "   • Or use the free hosted demo: https://huggingface.co/spaces/yisol/IDM-VTON"
            )
    else:
        raise RuntimeError("No CUDA GPU found.")
except Exception:
    subprocess.run(["nvidia-smi"], check=False)
    raise RuntimeError(
        "CUDA GPU required. In Colab: Runtime → Change runtime type → T4 GPU (or A100/L4)."
    )

print(f"\nPython {sys.version.split()[0]} | PyTorch {torch.__version__}")
print(f"Working directory: {WORK_DIR}")

In [ ]:
# @title 2) Optional — cache models on Google Drive (skip re-downloads)
USE_DRIVE_CACHE = False  # @param {type:"boolean"}
DRIVE_CACHE_DIR = "/content/drive/MyDrive/idm-vton-cache"  # @param {type:"string"}

if USE_DRIVE_CACHE:
    if "google.colab" not in sys.modules:
        print("Google Drive mounting is only available in Colab.")
    else:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
        os.environ["HF_HOME"] = DRIVE_CACHE_DIR
        os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(DRIVE_CACHE_DIR, "hub")
        print(f"HF cache → {DRIVE_CACHE_DIR}")
else:
    print("Drive cache disabled. Models download to the Colab session (lost on disconnect).")

In [ ]:
# @title 3) Download official Hugging Face Space (code + parsing checkpoints)
from pathlib import Path

REQUIRED_CKPT = [
    "ckpt/densepose/model_final_162be9.pkl",
    "ckpt/humanparsing/parsing_atr.onnx",
    "ckpt/humanparsing/parsing_lip.onnx",
    "ckpt/openpose/ckpts/body_pose_model.pth",
]

work_path = Path(WORK_DIR)
missing = [p for p in REQUIRED_CKPT if not (work_path / p).exists()]

if missing:
    print("Downloading yisol/IDM-VTON Space (first run only)...")
    !pip install -q "huggingface_hub[cli,hf_transfer]>=0.25.0"
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id="yisol/IDM-VTON",
        repo_type="space",
        local_dir=WORK_DIR,
        local_dir_use_symlinks=False,
    )
    print("Download complete.")
else:
    print(f"Using cached install at {WORK_DIR}")

import os
os.chdir(WORK_DIR)
print(f"cwd: {os.getcwd()}")

still_missing = [p for p in REQUIRED_CKPT if not Path(p).exists()]
if still_missing:
    raise FileNotFoundError(f"Missing checkpoints: {still_missing}")
print("All parsing checkpoints present.")

In [ ]:
# @title 4) Install dependencies (inference-only, Colab-safe)
import subprocess
import sys

# Only packages needed to run app.py — skip basicsr/pycocotools from Space requirements
# (they fail to build on Colab PyTorch 2.11 and are not used at inference time).
CORE_DEPS = [
    "diffusers==0.25.0",
    "transformers==4.36.2",
    "accelerate==0.26.1",
    "gradio==4.24.0",
    "einops==0.7.0",
    "onnxruntime==1.16.2",
    "opencv-python-headless>=4.7.0",
    "cloudpickle",
    "omegaconf",
    "fvcore",
    "av",
    "config",
    "huggingface_hub>=0.25.0",
    "scikit-image",
    "matplotlib",
    "tqdm",
    "pycocotools",  # binary wheel; needed by detectron2/densepose
]

failed = []
for pkg in CORE_DEPS:
    print(f"Installing {pkg} ...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        failed.append(pkg)
        tail = (result.stderr or result.stdout or "").strip().splitlines()
        print(f"  ⚠️  failed: {tail[-1] if tail else 'unknown error'}")

if failed:
    raise RuntimeError(
        "Required packages failed to install: "
        + ", ".join(failed)
        + "\nRe-run this cell, or Runtime → Restart session → Run all."
    )

import torch
print(f"\nReady — PyTorch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")

In [ ]:
# @title 5) Configure Detectron2 for Colab Python
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def detectron2_works() -> bool:
    try:
        from detectron2.data.detection_utils import convert_PIL_to_numpy  # noqa: F401
        return True
    except Exception as exc:
        print(f"detectron2 not ready yet: {exc}")
        return False


if not detectron2_works():
    print("Building detectron2 for this Colab runtime (3–8 min, one-time)...")
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "ninja-build"],
        check=True,
    )

    bundled = ROOT / "detectron2"
    if bundled.exists() and not (ROOT / "detectron2_bundled").exists():
        bundled.rename(ROOT / "detectron2_bundled")
        print("Moved bundled detectron2 (py3.9 binary) → detectron2_bundled")

    env = os.environ.copy()
    env["MAX_JOBS"] = "2"
    build = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "git+https://github.com/facebookresearch/detectron2.git",
            "--no-build-isolation",
        ],
        env=env,
        capture_output=True,
        text=True,
    )
    if build.returncode != 0:
        tail = (build.stderr or build.stdout or "").strip().splitlines()[-8:]
        raise RuntimeError(
            "detectron2 build failed. Try Runtime → Restart session → Run all.\n"
            + "\n".join(tail)
        )

if not detectron2_works():
    raise RuntimeError("detectron2 import still failing after build.")

print("Detectron2 OK")

In [ ]:
# @title 6) Hugging Face login (optional — public model, usually not required)
from huggingface_hub import HfFolder, login

if HfFolder.get_token():
    print("Already logged in to Hugging Face.")
else:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
        login(token=token, add_to_git_credential=False)
        print("Logged in via Colab secret HF_TOKEN.")
    except Exception:
        print(
            "No HF token found. Continuing without login (public weights load automatically).\n"
            "To add a token: Colab → 🔑 Secrets → HF_TOKEN, or uncomment login() below."
        )
        # login()  # uncomment for interactive login

In [ ]:
# @title 7) Patch app for Colab & launch Gradio (public share link)
import re
from pathlib import Path

LOW_VRAM_MODE = False  # @param {type:"boolean"}

app_path = Path("app.py")
if not app_path.exists():
    app_path = Path("gradio_demo/app.py")
if not app_path.exists():
    raise FileNotFoundError("Could not find app.py — re-run the download cell.")

text = app_path.read_text()
text = text.replace("import spaces\n", "")
text = re.sub(r"@spaces\.GPU\s*\n", "", text)

if LOW_VRAM_MODE:
    offload = (
        "    pipe.enable_sequential_cpu_offload()\n"
        "    # Low-VRAM mode: skip pipe.to(device) calls below\n"
    )
    text = text.replace(
        "    pipe.to(device)\n    pipe.unet_encoder.to(device)",
        offload,
    )
    print("Low-VRAM mode enabled (slower, may help T4 — quality can vary).")

launch_line = "image_blocks.launch(share=True, server_name='0.0.0.0', server_port=7860, debug=True)"
if "image_blocks.launch()" in text:
    text = text.replace("image_blocks.launch()", launch_line)
elif "demo.launch()" in text:
    text = text.replace("demo.launch()", launch_line.replace("image_blocks", "demo"))

colab_app = Path("colab_app.py")
colab_app.write_text(text)
print(f"Patched app written to {colab_app.resolve()}")
print("Launching Gradio — copy the 'Running on public URL' link below.\n")

!python colab_app.py

## Tips for best results
- **Person photo:** front-facing, well-lit, plain background, upper body visible.
- **Garment photo:** flat-lay or mannequin product shots work better than photos of someone wearing the item.
- Enable **auto-generated mask** and **auto-crop** if the photo is not already framed for upper-body try-on.
- Increase **denoising steps** (30–40) in Advanced Settings for sharper output.

## Troubleshooting
| Issue | Fix |
|---|---|
| `Getting requirements to build wheel` / pip build error | Fixed in cell 4 — re-run all cells from the latest notebook (skips `basicsr` and other training-only deps) |
| `CUDA out of memory` / session killed | Switch to **A100/L4** (Colab Pro+), enable `LOW_VRAM_MODE` in cell 7, or use the [hosted demo](https://huggingface.co/spaces/yisol/IDM-VTON) |
| Gradio link expired | Re-run the launch cell for a fresh public URL |
| Detectron2 build fails | Runtime → **Restart session**, then run all cells again (cell 5 installs `ninja-build` and compiles detectron2) |
| Slow re-downloads every session | Set `USE_DRIVE_CACHE = True` in cell 2 |
| `Cannot copy out of meta tensor` (low VRAM) | Disable `LOW_VRAM_MODE` and use a larger GPU |

## License
IDM-VTON code and weights are under [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/) (non-commercial use).